In [ ]:
# %matplotlib inline
# %reload_ext autoreload
# %autoreload 2

In [ ]:
import numpy as np
import mne
import mne_connectivity
from mne_connectivity import phase_slope_index, seed_target_indices, spectral_connectivity_epochs
from glob import glob
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
from scipy.stats import zscore, linregress, ttest_ind, ttest_rel, ttest_1samp
import pandas as pd
from mne.preprocessing.bads import _find_outliers
import fooof
# Import plotting functions
from fooof.plts.spectra import plot_spectrum, plot_spectra
# Import the FOOOF object
from fooof import FOOOF
from fooof import FOOOFGroup
import os 
import joblib
import statsmodels.formula.api as smf
import statsmodels.api as sm
import pickle
from tqdm import tqdm
from IPython.display import clear_output
import warnings 

# I only want to see warnings once
warnings.filterwarnings('ignore')

from joblib import delayed, Parallel
from statsmodels.stats import multitest

Note: If you have installed this package in editable form on Minerva, you must append the local path! This is because Minerva requires that you point your package installs away from the local directory for space reasons, but editable packages have to be installed locally.

In [ ]:
import sys
sys.path.append('/hpc/users/qasims01/resources/LFPAnalysis')

In [ ]:
from LFPAnalysis import lfp_preprocess_utils, sync_utils, analysis_utils, nlx_utils, oscillation_utils, statistics_utils


## First, get the behavior data

This is from another notebook with a kernel and environment specifically for beahvior

In [ ]:
base_dir = '/sc/arion' # this is the root directory for most un-archived data and results
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'

# Load the demographics data to get information about who has run the task
demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion==1) & (demo_data.Minerva==1)]

# Let's get rid of Iowa patients for now 

# subj_df = subj_df[subj_df.Site!='UI']
subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]


days = ['Day1', 'Day2']
subj_ids = subj_df.MSSMCode.values
subj_formats = subj_df.format.values
subj_sites = subj_df.Site.values
subj_days = subj_df.Day2.values

In [ ]:
# load behavioral data
learn_df = pd.read_csv(f'{load_dir}/learn_df_RWH.csv')
combined_df = pd.read_csv(f'{load_dir}/full_df_RWH.csv')



The following function provides a parallelized function for extracting time-resolved data from the TFRs (averaged across band). 

In [ ]:
analysis_evs = ['feedback_start']

In [ ]:
%%time
progress_bar = tqdm(subj_ids, ascii=True, desc='Saving TFR dataframes for time-resolved data')

power_epochs = {f'{a}': {f'{b}': np.nan for b in analysis_evs} for a in subj_ids[0:1]}

for subj_id in progress_bar:
    for day in ['Day1']:
        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')
        for event in analysis_evs:

            filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
            if not os.path.exists(f'{filepath}/{event}/plots'):
                os.makedirs(f'{filepath}/{event}/plots')
            if not os.path.exists(f'{filepath}/{event}/dfs'):
                os.makedirs(f'{filepath}/{event}/dfs')

            power_epochs[subj_id][event] = mne.time_frequency.read_tfrs(f'{filepath}/{event}-tfr.h5')[0]

            epochs_to_analyze = learn_df[(learn_df.participant==subj_id)].trials.values - 1

            power_epochs[subj_id][event] = power_epochs[subj_id][event][epochs_to_analyze.astype(int)]

            power_epochs[subj_id][event].metadata = learn_df[(learn_df.participant==subj_id)]
            
#             # If you want to crop to a specific time, do so here: 
            
            
#             bands=['hfa']
            
#             # Run the regression code in parallel
#             Parallel(n_jobs=-1, verbose=1)(delayed(statistic_utils.tfr_to_bandavg_timeseries)(chan,
#                                                                               power_epochs[subj_id][event], 
#                                                                               subj_id, 
#                                                                               elec_df, 
#                                                                               event,
#                                                                               bands) for chan in power_epochs[subj_id][event].ch_names)


In [ ]:
elec_df[elec_df.salman_region.isin(['AMY'])].index.values

In [ ]:
elec_df.Channel

In [ ]:
day = 'Day1'

progress_bar = tqdm(subj_ids, ascii=True, desc='Performing time-resolved regression')
freqs = np.logspace(*np.log10([2, 200]), num=30)
event = 'feedback_start'

freq_dict = {'theta':[2, 8], 
            'alpha':[8, 13],
            'beta': [13, 30], 
            'hfa': [70, 200]}


Let's make an attempt to decode memory as a function of time in HFA during both the encoding and recall periods to identify when memory information is actually being processed in each individual brain and electrode

In [ ]:
# from sklearn.linear_model import LogisticRegression
# from sklearn.model_selection import cross_val_score
# from sklearn.impute import SimpleImputer
# from joblib import Parallel, delayed
# import numpy as np

# def decode_memory_lr_with_imputation(timeseries_data, memory_labels, n_jobs=-1):
#     n_trials, n_timepoints = timeseries_data.shape
#     performance_scores = np.zeros(n_timepoints)
#     decode_memory_lr_with_imputation
#     # Imputer to replace NaN with the mean of each timepoint
#     imputer = SimpleImputer(strategy='mean')

#     def decode_timepoint(t):
#         # Extract the data for the current timepoint across all trials
#         timepoint_data = timeseries_data[:, t].reshape(-1, 1)

#         # Impute missing values (NaN)
#         timepoint_data = imputer.fit_transform(timepoint_data)

#         # Logistic Regression classifier with class weighting to handle imbalance
#         clf = LogisticRegression(class_weight='balanced', solver='liblinear')

#         # Use cross-validation with AUC-ROC scoring
#         scores = cross_val_score(clf, timepoint_data, memory_labels, cv=5, scoring='roc_auc', n_jobs=n_jobs)

#         return np.mean(scores)
    
#     # Use joblib to parallelize decoding over timepoints
#     performance_scores = Parallel(n_jobs=n_jobs)(delayed(decode_timepoint)(t) for t in range(n_timepoints))

#     return performance_scores

In [ ]:
# find memory sensitive electrodes using permutation cluster test and hfa-smoothed data
from mne.stats import permutation_cluster_test
from scipy.stats import ttest_ind
def stat_fun(a, b):
      return ttest_ind(a, b, nan_policy='omit')[0]


In [ ]:
progress_bar = tqdm(subj_ids, ascii=True, desc='Performing time-resolved regression')
freqs = np.logspace(*np.log10([2, 200]), num=30)
event = 'feedback_start'

freq_dict = {'theta':[2, 8], 
            'alpha':[8, 13],
            'beta': [13, 30], 
            'hfa': [70, 200]}

chan_dfs = []

for band in ['hfa']:        
    for subj_id in subj_ids:
        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')

        filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
        tfr_data = mne.time_frequency.read_tfrs(f'{filepath}/{event}-tfr.h5')[0]
        
        memory_df = combined_df[(combined_df.condition=='Day1') & (combined_df.participant==subj_id)].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits']]
        memory_df.rename(columns={'trials_gamble':'trials'}, inplace=True)
        memory_df['phit'] = memory_df.apply(lambda x: 1 if x.hits==True else 0, axis=1)
        epochs_to_analyze = memory_df.trials - 1
        
        trunc_epochs = tfr_data[epochs_to_analyze.astype(int)]

        relevant_channels = elec_df[elec_df.salman_region.isin(['HPC', 'AMY', 'dmPFC', 'OFC', 'ACC'])].index.values
        
        for channel in relevant_channels:
            label = elec_df.iloc[channel].label
            # timeseries = np.nanmean(trunc_epochs.copy().pick(label).data[:, 0,
            #                                      np.where((freqs>=freq_dict[band][0]) & (freqs<freq_dict[band][1]))[0],
            #                                      :], axis=1)
            
#             # smooth HFA: 
#             if band == 'hfa':
#                 win_len = int(50  / (1000/tfr_data.info['sfreq']))
#                 slide_len = int(24 / (1000/tfr_data.info['sfreq']))
#                 slices = np.lib.stride_tricks.sliding_window_view(np.arange(timeseries.shape[-1]),
#                                                                   win_len)[::slide_len]
#                 midpoints = np.ceil(np.mean(slices, axis=1))
#                 # Smooth the timeseries (easier to do here than to store smoothed data)
#                 sig = np.zeros([timeseries.shape[0], slices.shape[0]])
#                 for stride in range(slices.shape[0]):
#                     sig[:, stride] = np.nanmean(timeseries[:, slices[stride]], axis=1)

#                 times = midpoints * (1000/tfr_data.info['sfreq'])
#             else:
#                 sig = timeseries
#                 times = tfr_data.times
            bad_memory = trunc_epochs.copy().pick(label).data[memory_df.phit==0, 0, : :]
            good_memory = trunc_epochs.copy().pick(label).data[memory_df.phit==1, 0, : :]
            T_obs, clusters, cluster_p_values, H0 = permutation_cluster_test([good_memory, bad_memory],
                                                                             n_permutations=1000, stat_fun = stat_fun,
                                                                             n_jobs=-1, verbose=False,
                                                                             out_type="mask",)
            
            if any(cluster_p_values<=0.05):
                
#                 fig, (ax, ax2) = plt.subplots(2, 1, figsize=(8, 4))
#                 ax.set_title("Channel : " + label)
#                 ax.plot(
#                     times,
#                     np.nanmean(good_memory, axis=0) - np.nanmean(bad_memory, axis=0),
#                     label="ERF Contrast (Event 1 - Event 2)",
#                 )
#                 # ax.set_ylabel("MEG (T / m)")
#                 ax.legend()

#                 for i_c, c in enumerate(clusters):
#                     c = c[0]
#                     if cluster_p_values[i_c] <= 0.05:
#                         h = ax2.axvspan(times[c.start], times[c.stop - 1], color="r", alpha=0.3)
#                     else:
#                         ax2.axvspan(times[c.start], times[c.stop - 1], color=(0.3, 0.3, 0.3), alpha=0.3)

#                 hf = plt.plot(times, T_obs, "g")
#                 ax2.legend((h,), ("cluster p-value < 0.05",))
#                 ax2.set_xlabel("time (ms)")
#                 ax2.set_ylabel("f-values")
                
#             performance_scores =  decode_memory_lr_with_imputation(timeseries, memory_df.phit.values, n_jobs=-1)
                test = pd.DataFrame(columns=['channel', 'subj', 'label', 'region'])
            
#             test['ts'] = np.arange(timeseries.shape[-1]) * (1000/tfr_data.info['sfreq'])
#             test['AUC'] = performance_scores
                test['channel'] = [channel]
                test['subj'] = [subj_id]
                test['label'] = [label]
                test['region'] = [elec_df.iloc[channel].salman_region]
                # test['band'] = band
            
                chan_dfs.append(test)
            
all_dfs = pd.concat(chan_dfs)            

In [ ]:
all_dfs.region.value_counts()

In [ ]:
from scipy.stats import binomtest

binomtest(12, 96, 0.05)

In [ ]:
times = midpoints * (1000/tfr_data.info['sfreq'])
            
fig, (ax, ax2) = plt.subplots(2, 1, figsize=(8, 4))
ax.set_title("Channel : " + label)
ax.plot(
    times,
    good_memory.mean(axis=0) - bad_memory.mean(axis=0),
    label="ERF Contrast (Event 1 - Event 2)",
)
# ax.set_ylabel("MEG (T / m)")
ax.legend()

for i_c, c in enumerate(clusters):
    c = c[0]
    if cluster_p_values[i_c] <= 0.05:
        h = ax2.axvspan(times[c.start], times[c.stop - 1], color="r", alpha=0.3)
    else:
        ax2.axvspan(times[c.start], times[c.stop - 1], color=(0.3, 0.3, 0.3), alpha=0.3)

hf = plt.plot(times, T_obs, "g")
ax2.legend((h,), ("cluster p-value < 0.05",))
ax2.set_xlabel("time (ms)")
ax2.set_ylabel("f-values")

In [ ]:
# Step 1: Sort and organize by 'region' and 'subj'
df_sorted = all_dfs.sort_values(by=['subj', 'region', 'channel'])

# Step 2: Create a pivot table with 'ts' as x-axis, 'channel' as y-axis, and 'AUC' as values
df_pivot = df_sorted.pivot_table(index='channel', columns='ts', values='AUC', aggfunc='mean')

# Step 3: Create the heatmap
plt.figure(figsize=(12, 8))  # Adjust size as needed
df_pivot[df_pivot<=0.5] = 0.5
ax = sns.heatmap(df_pivot, cmap='Spectral_r', cbar_kws={'label': 'AUC'})

# Step 4: Add labels and a title
ax.set_title('AUC Heatmap Across Timepoints and Channels', fontsize=16)
ax.set_xlabel('Timepoint (ts)', fontsize=12)
ax.set_ylabel('Channel', fontsize=12)

# Show the plot
plt.tight_layout()
plt.show()

In [ ]:
sns.lineplot(data=all_dfs, x='ts', y='AUC', errorbar='se')
plt.hlines(0.5, 0, 1500, 'r')

In [ ]:
day = 'Day1'

In [ ]:

progress_bar = tqdm(subj_ids, ascii=True, desc='Performing time-resolved regression')
freqs = np.logspace(*np.log10([2, 200]), num=30)
event = 'feedback_start'

freq_dict = {'theta':[2, 8], 
            'alpha':[8, 13],
            'beta': [13, 30], 
            'hfa': [70, 200]}

for band in ['theta', 'beta', 'hfa']:
    if band == 'theta':
        rois = ['dlPFC', 'OFC']
    elif band == 'beta': 
        rois = ['ACC'] 
    elif band == 'hfa': 
        rois = ['AMY']
    all_rpe_channels  = [] 
        
    for subj_id in progress_bar:
        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')

        filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
        tfr_data = mne.time_frequency.read_tfrs(f'{filepath}/{event}-tfr.h5')[0]
        regressors = learn_df[learn_df.participant==subj_id][['rpe', 'DPRIME']]
        all_chan_df = []

        relevant_channels = elec_df[elec_df.salman_region.isin(rois)].index.values
        
        if len(relevant_channels) == 0:
            print('no relevant channels')
            continue
            
        for channel in relevant_channels:
            label = elec_df.iloc[channel].label
            timeseries = np.nanmean(tfr_data.copy().pick(label).data[:, 0,
                                                 np.where((freqs>=freq_dict[band][0]) & (freqs<freq_dict[band][1]))[0],
                                                 :], axis=1)
            # timeseries[timeseries>=10] = np.nan
            test = statistics_utils.time_resolved_regression_single_channel(timeseries, 
                                                  regressors,standardize=True, smooth=True, 
                                                                            win_len=50, slide_len=25) # 50, 25 for HFA
            test['channel'] = channel
            test['subj'] = subj_id
            test['label'] = label
            test['region'] = elec_df.iloc[channel].salman_region
            test['band'] = band
            all_chan_df.append(test)  

        all_chan_df = pd.concat(all_chan_df).reset_index()
        all_rpe_channels.append(all_chan_df)

    all_rpe_channels = pd.concat(all_rpe_channels)
    all_rpe_channels.to_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{event}_time_res_regress', index=False)


Now run a version that splits the OFC data by memory

In [ ]:

progress_bar = tqdm(subj_ids, ascii=True, desc='Performing time-resolved regression')
freqs = np.logspace(*np.log10([2, 200]), num=30)
event = 'feedback_start'

freq_dict = {'theta':[2, 8], 
            'alpha':[8, 13],
            'beta': [13, 30], 
            'hfa': [70, 200]}

for band in ['hfa']:
    if band == 'theta':
        rois = ['dlPFC', 'OFC']
    elif band == 'beta': 
        rois = ['ACC'] 
    elif band == 'hfa': 
        rois = ['OFC']
    all_rpe_channels  = [] 
        
    for subj_id in progress_bar:
        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')

        filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
        tfr_data = mne.time_frequency.read_tfrs(f'{filepath}/{event}-tfr.h5')[0]
        regressors = learn_df[learn_df.participant==subj_id][['rpe', 'DPRIME']]
        all_chan_df = []

        relevant_channels = elec_df[elec_df.salman_region.isin(rois)].index.values
        
        if len(relevant_channels) == 0:
            print('no relevant channels')
            continue
            
        for channel in relevant_channels:
            label = elec_df.iloc[channel].label
            
            # Split the data into recalled and non-recalled: 
            memory_df = combined_df[(combined_df.condition == 'Day1') & (combined_df.participant == subj_id)].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits']]
            rec_ix = memory_df[memory_df.hits==True].trials_gamble.values.astype(int) - 1
            
            
            timeseries_rec = np.nanmean(tfr_data.copy().pick(label).data[:, 0,
                                                 np.where((freqs>=freq_dict[band][0]) & (freqs<freq_dict[band][1]))[0],
                                                 :][rec_ix,:], axis=1)
            
            # timeseries[timeseries>=10] = np.nan
            test1 = statistics_utils.time_resolved_regression_single_channel(timeseries_rec, 
                                                  regressors.iloc[rec_ix],standardize=True, smooth=True, 
                                                                            win_len=50, slide_len=25) # 50, 25 for HFA
            test['channel'] = channel
            test1['subj'] = subj_id
            test1['label'] = label
            test1['region'] = elec_df.iloc[channel].salman_region
            test1['band'] = band
            test1['hit'] = True
            
            all_chan_df.append(test1)  
            
            nrec_ix = memory_df[memory_df.hits==False].trials_gamble.values.astype(int) - 1
            
            timeseries_nrec = np.nanmean(tfr_data.copy().pick(label).data[:, 0,
                                                 np.where((freqs>=freq_dict[band][0]) & (freqs<freq_dict[band][1]))[0],
                                                 :][nrec_ix,:], axis=1)
            
            test2 = statistics_utils.time_resolved_regression_single_channel(timeseries_nrec, 
                                                  regressors.iloc[nrec_ix],standardize=True, smooth=True, 
                                                                            win_len=50, slide_len=25) # 50, 25 for HFA
            test2['channel'] = channel
            test2['subj'] = subj_id
            test2['label'] = label
            test2['region'] = elec_df.iloc[channel].salman_region
            test2['band'] = band
            test2['hit'] = False
            
            all_chan_df.append(test2)  

        all_chan_df = pd.concat(all_chan_df).reset_index()
        all_rpe_channels.append(all_chan_df)

    all_rpe_channels = pd.concat(all_rpe_channels)
    all_rpe_channels.to_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{event}_time_res_regress_rec_split_OFC', index=False)


In [ ]:
band = 'hfa' 
event = 'feedback_start'
all_rpe_channels = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{event}_time_res_regress_rec_split_OFC')
                        
                        

In [ ]:
band = 'hfa' 
event = 'feedback_start'
all_rpe_channels = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{event}_time_res_regress_rec_split_OFC')
                        
          
hit_df = all_rpe_channels[(all_rpe_channels['index']=='rpe') & (all_rpe_channels['region']) & (all_rpe_channels['hit'] == True)]
miss_df = all_rpe_channels[(all_rpe_channels['index']=='rpe') & (all_rpe_channels['region'])& (all_rpe_channels['hit'] == False)]
contrast_df = hit_df.copy() 
contrast_df['Original_Estimate'] = hit_df.Original_Estimate.values - miss_df.Original_Estimate.values
first_half_df = contrast_df[contrast_df.ts<=750].groupby(['subj', 'label']).mean(numeric_only=True).reset_index()
second_half_df = contrast_df[contrast_df.ts>750].groupby(['subj', 'label']).mean(numeric_only=True).reset_index()
half_quotient = first_half_df.Original_Estimate - second_half_df.Original_Estimate

In [ ]:
half_q_df = first_half_df.copy()
half_q_df['half_q'] = half_quotient
half_q_df = half_q_df[['subj', 'label', 'half_q']]
half_q_df.rename(columns={'subj':'participant'}, inplace=True)


In [ ]:
half_q_df.to_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/half_q_df', index=False)

In [ ]:
all_rpe_channels

In [ ]:
half_quotient

In [ ]:
np.nanmean(half_quotient[half_quotient<=0])

In [ ]:
sns.regplot(x=first_half_df.Original_Estimate, 
            y=second_half_df.Original_Estimate)
            

In [ ]:
sns.distplot(half_quotient)

In [ ]:
band = 'hfa'
analysis_evs = 'feedback_start'
gamble_model_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{analysis_evs}_elec_level_df_intx')




In [ ]:
sns.regplot(x=half_quotient, 
            y=gamble_model_df[gamble_model_df.region=='OFC']['phit:zrpe'])

So there is no preference for early vs. late responding. 

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 8), dpi=300)
sns.despine()
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_{band}_rpe_time_resolved_rec_split_OFC.pdf') as pdf:
    sns.pointplot(data=all_rpe_channels[(all_rpe_channels['index']=='rpe') & (all_rpe_channels['region']) & (all_rpe_channels['hit'] == True)],
         x='ts', y='Original_Estimate', errorbar='se', color='g', ax=ax)
    sns.pointplot(data=all_rpe_channels[(all_rpe_channels['index']=='rpe') & (all_rpe_channels['region'])& (all_rpe_channels['hit'] == False)],
         x='ts', y='Original_Estimate', errorbar='se', color='r', ax=ax)
    ax.set(ylim=[-0.08, 0.08])
    pdf.savefig()
    

In [ ]:
combined_df.trials.min()

In [ ]:
# # Assuming your dataframe is called 'df'
# all_rpe_channels['any_sig'] = 0
# all_rpe_channels['any_sig'] = (all_rpe_channels.groupby(['subj', 'label'])['P_Value']
#                    .transform(lambda x: np.any(x <= (0.05/27)))
#                    .astype(int))
# # all_rpe_channels['best_ts'] = all_rpe_channels.groupby(['subj', 'label']).apply(
# #     lambda x: x.loc[x['P_Value'].idxmin(), 'ts']
# # ).reset_index(level=[0,1], drop=True)

In [ ]:
# fig, ax = plt.subplots(1, 1, figsize=(6, 4), dpi=300)
# sns.despine()
# sns.pointplot(data=all_rpe_channels[(all_rpe_channels['index']=='rpe') & (all_rpe_channels['region'].isin(['dmPFC']))],
#      x='ts', y='Original_Estimate', errorbar='se', hue='any_sig', ax=ax)


In [ ]:
# # plt.hist(all_rpe_channels[(all_rpe_channels['index']=='rpe') & (all_rpe_channels.region.isin(['AMY']))].best_ts)
# # plt.hist(all_rpe_channels[(all_rpe_channels['index']=='rpe') & (all_rpe_channels.region.isin(['HPC']))].best_ts)
# fig, ax = plt.subplots(1, 1, figsize=(5, 4), dpi=300)

# sns.histplot(data=all_rpe_channels[(all_rpe_channels['index']=='rpe') & (all_rpe_channels.region.isin(['OFC', 'AINS']))].reset_index(), x='best_ts', hue='region', 
#          element='step', stat="density", common_norm=False, ax=ax)
    

In [ ]:
band = 'theta'
all_rpe_channels = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{event}_time_res_regress')


In [ ]:
# fig, ax = plt.subplots(1, 1, figsize=(10, 6), dpi=300)
# sns.despine()
# with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_{band}_Intercept_time_resolved.pdf') as pdf:
#     sns.lineplot(data=all_rpe_channels[(all_rpe_channels['index']=='Intercept')],
#          x='ts', y='Original_Estimate', errorbar='se', hue='region', ax=ax)
#     ax.set(ylim=[-0.1, 0.1])
#     pdf.savefig()
    

In [ ]:
# Load results of single-electrode regression
band = 'theta'
gamble_model_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{event}_rpe_pm_phit_eleclevel_df')


gamble_model_df[gamble_model_df.rpe<-1.96][gamble_model_df.region=='dmPFC']

In [ ]:
filtered_df = all_rpe_channels[(all_rpe_channels['index']=='rpe')]
filtered_df = filtered_df[filtered_df['subj'].isin(gamble_model_df[gamble_model_df.rpe<-1.96][gamble_model_df.region=='dmPFC'].participant
                                                  ) & filtered_df['label'].isin(gamble_model_df[gamble_model_df.rpe<-1.96][gamble_model_df.region=='dmPFC'].channel)]

fig, ax = plt.subplots(1, 1, figsize=(8, 8), dpi=300)
sns.despine()
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_{band}_rpe_time_resolved_sigdmpfc.pdf') as pdf:

    sns.pointplot(data=filtered_df,
         x='ts', y='Original_Estimate', errorbar='se', ax=ax, color='red')
    
    pdf.savefig()

In [ ]:
filtered_df = all_rpe_channels[(all_rpe_channels['index']=='rpe')]
filtered_df = filtered_df[filtered_df['subj'].isin(gamble_model_df[gamble_model_df.rpe<-1.96][gamble_model_df.region=='OFC'].participant
                                                  ) & filtered_df['label'].isin(gamble_model_df[gamble_model_df.rpe<-1.96][gamble_model_df.region=='OFC'].channel)]

fig, ax = plt.subplots(1, 1, figsize=(8, 8), dpi=300)
sns.despine()
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_{band}_rpe_time_resolved_sigofc.pdf') as pdf:

    sns.pointplot(data=filtered_df,
         x='ts', y='Original_Estimate', errorbar='se', ax=ax, color='yellow')
    
    pdf.savefig()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 8), dpi=300)
sns.despine()
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_{band}_rpe_time_resolved.pdf') as pdf:
    sns.pointplot(data=all_rpe_channels[(all_rpe_channels['index']=='rpe') & (all_rpe_channels['region'].isin(['AMY','OFC']))],
         x='ts', y='Original_Estimate', errorbar='se', hue='region', ax=ax)
    # ax.set(ylim=[-0.075, 0.075])
    pdf.savefig()
    

In [ ]:
band = 'hfa'
all_rpe_channels = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{event}_time_res_regress')


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 8), dpi=300)
sns.despine()
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_{band}_rpe_time_resolved.pdf') as pdf:
    sns.pointplot(data=all_rpe_channels[(all_rpe_channels['index']=='rpe') & (all_rpe_channels['region'])],
         x='ts', y='Original_Estimate', errorbar='se', hue='region', ax=ax)
    ax.set(ylim=[-0.05, 0.05])
    pdf.savefig()
    

In [ ]:
# fig, ax = plt.subplots(1, 1, figsize=(10, 6), dpi=300)
# sns.despine()
# with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_{band}_rpe_time_resolved.pdf') as pdf:
#     sns.pointplot(data=all_rpe_channels[(all_rpe_channels['index']=='rpe') & (all_rpe_channels['region'].isin(['HPC','dmPFC']))],
#          x='ts', y='Original_Estimate', errorbar='se', hue='region', ax=ax)
#     ax.set(ylim=[-0.1, 0.1])
#     pdf.savefig()
    

In [ ]:
# fig, ax = plt.subplots(1, 1, figsize=(10, 6), dpi=300)
# sns.despine()
# # with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_hfa_rpe_eleclevel_time_resolved.pdf') as pdf:
# sns.pointplot(data=all_rpe_channels[(all_rpe_channels['index']=='rpe') & (all_rpe_channels.region.isin(['AINS']))],
#          x='ts', y='Original_Estimate', errorbar='se', hue='region', ax=ax)
#     # pdf.savefig()

In [ ]:
df_max = all_rpe_channels[(all_rpe_channels['index']=='rpe') & (all_rpe_channels['region'].isin(['AMY','OFC']))].groupby(['subj', 'label', 'region']).apply(
    lambda x: x.loc[x['P_Value'].idxmin(), 'ts']).reset_index().rename(columns={0:'peak_ts'})

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 4), dpi=300)
sns.despine()
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_hfa_rpe_eleclevel_max_ts.pdf') as pdf:
    sns.histplot(data=df_max, 
                 x='peak_ts', hue='region', 
             element='step', stat="density", common_norm=False, ax=ax)
    pdf.savefig()
    


In [ ]:
ttest_ind(df_max[df_max.region=='AMY'].peak_ts,
          df_max[df_max.region=='OFC'].peak_ts)

In [ ]:
all_rpe_channels[(all_rpe_channels['index']=='rpe') & (all_rpe_channels['region'].isin(['AMY','OFC']))]




In [ ]:

progress_bar = tqdm(subj_ids, ascii=True, desc='Performing time-resolved regression')
freqs = np.logspace(*np.log10([2, 200]), num=30)
event = 'feedback_start'

freq_dict = {'theta':[2, 8], 
            'alpha':[8, 13],
            'beta': [13, 30], 
            'hfa': [70, 200]}

for band in ['hfa']:
    all_rpe_channels  = [] 
        
    for subj_id in progress_bar:
        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')

        filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
        tfr_data = mne.time_frequency.read_tfrs(f'{filepath}/{event}-tfr.h5')[0]
        regressors = learn_df[learn_df.participant==subj_id][['rpe', 'DPRIME']]
        all_chan_df = []

        relevant_channels = elec_df[elec_df.salman_region.isin(['AMY', 'HPC', 'OFC', 'dmPFC'])].index.values
        
        for channel in relevant_channels:
            label = elec_df.iloc[channel].label
            timeseries = np.nanmean(tfr_data.copy().pick(label).data[:, 0,
                                                 np.where((freqs>=freq_dict[band][0]) & (freqs<freq_dict[band][1]))[0],
                                                 :], axis=1)

            test = statistics_utils.time_resolved_regression_single_channel(timeseries, 
                                                  regressors,standardize=True, smooth=True, 
                                                                            win_len=50, slide_len=25) # 50, 25 for HFA
            test['channel'] = channel
            test['subj'] = subj_id
            test['label'] = label
            test['region'] = elec_df.iloc[channel].salman_region
            test['band'] = band
            all_chan_df.append(test)  

        all_chan_df = pd.concat(all_chan_df).reset_index()
        all_rpe_channels.append(all_chan_df)

    all_rpe_channels = pd.concat(all_rpe_channels)
    all_rpe_channels.to_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{event}_time_res_regress_rpe_splitphit', index=False)


In [ ]:
#  %%time
# progress_bar = tqdm(subj_ids, ascii=True, desc='Saving TFR dataframes for time-resolved data')
# freqs = np.logspace(*np.log10([2, 200]), num=30)
# event = 'feedback_start'
# all_phit_channels  = [] 
# for subj_id in progress_bar:
#     elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')
    
#     filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
#     tfr_data = mne.time_frequency.read_tfrs(f'{filepath}/{event}-tfr.h5')[0]
#     all_hfa = np.nanmean(tfr_data.data[:, :, np.where(freqs>70)[0], :], axis=2)
                
#     memory_df = combined_df[(combined_df.condition=='Day1') & (combined_df.participant==subj_id)].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits']]
#     memory_df.rename(columns={'trials_gamble':'trials'}, inplace=True)
#     memory_df['phit'] = memory_df.apply(lambda x: 1 if x.hits==True else -1, axis=1)
#     epochs_to_include = memory_df.trials.values.astype(int) - 1
    
#     regressors = memory_df[['phit']]
                
#     all_chan_df = []
    
#     for channel in range(all_hfa.shape[1]):
#         test = statistics_utils.time_resolved_regression(all_hfa[epochs_to_include, channel, :], 
#                                               regressors, win_len=100, 
#                              slide_len=25, 
#                              standardize=True, 
#                              sr=tfr_data.info['sfreq'])
#         test['channel'] = channel
#         test['subj'] = subj_id
#         test['label'] = elec_df.iloc[channel].label
#         test['region'] = elec_df.iloc[channel].salman_region
#         all_chan_df.append(test)  

#     all_chan_df = pd.concat(all_chan_df).reset_index()
#     relevant_channels = elec_df[elec_df.salman_region.isin(['HPC'])].index.values
#     relevant_channels_df = all_chan_df[all_chan_df.channel.isin(relevant_channels)]
#     all_phit_channels.append(relevant_channels_df)

# all_phit_channels = pd.concat(all_phit_channels)


In [ ]:

progress_bar = tqdm(subj_ids, ascii=True, desc='Performing time-resolved regression')
freqs = np.logspace(*np.log10([2, 200]), num=30)
event = 'feedback_start'

freq_dict = {'theta':[2, 8], 
            'alpha':[8, 13],
            'beta': [13, 30], 
            'hfa': [70, 200]}

for band in ['hfa']:
    all_phit_channels  = [] 
        
    for subj_id in progress_bar:
        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')

        filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
        tfr_data = mne.time_frequency.read_tfrs(f'{filepath}/{event}-tfr.h5')[0]

        memory_df = combined_df[(combined_df.condition=='Day1') & (combined_df.participant==subj_id)].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits']]
        memory_df.rename(columns={'trials_gamble':'trials'}, inplace=True)
        memory_df['phit'] = memory_df.apply(lambda x: 1 if x.hits==True else -1, axis=1)
        epochs_to_include = memory_df.trials.values.astype(int) - 1

        regressors = memory_df[['phit']]

        all_chan_df = []

        relevant_channels = elec_df[elec_df.salman_region.isin(['AMY', 'HPC', 'OFC', 'dmPFC'])].index.values
        
        for channel in relevant_channels:
            label = elec_df.iloc[channel].label
            timeseries = np.nanmean(tfr_data.copy().pick(label).data[:, 0,
                                                 np.where((freqs>=freq_dict[band][0]) & (freqs<freq_dict[band][1]))[0],
                                                 :], axis=1)
            timeseries = timeseries[epochs_to_include]
            test = statistics_utils.time_resolved_regression_single_channel(timeseries, 
                                                  regressors,standardize=True, smooth=True, 
                                                                            win_len=50, slide_len=25) # 50, 25 for HFA
            test['channel'] = channel
            test['subj'] = subj_id
            test['label'] = label
            test['region'] = elec_df.iloc[channel].salman_region
            test['band'] = band
            all_chan_df.append(test)  

        all_chan_df = pd.concat(all_chan_df).reset_index()
        all_phit_channels.append(all_chan_df)

    all_phit_channels = pd.concat(all_phit_channels)
    all_phit_channels.to_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{event}_time_res_regress_phit', index=False)


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 8), dpi=300)
sns.despine()
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_{band}_phit_time_resolved.pdf') as pdf:
    sns.pointplot(data=all_phit_channels[(all_phit_channels['index']=='phit') & (all_phit_channels['region'].isin(['AMY','HPC']))],
         x='ts', y='Original_Estimate', errorbar='se', hue='region', ax=ax)
    # ax.set(ylim=[-0.075, 0.075])
    pdf.savefig()
    

In [ ]:
test = 'llesion5-llesion6'


In [ ]:
'lesion' in test

In [ ]:
df_max = all_phit_channels.loc[all_phit_channels.groupby(['subj', 'label'])['phit'].idxmax()]


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 4), dpi=300)
sns.despine()
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_hfa_phit_eleclevel_max_ts.pdf') as pdf:
    sns.histplot(data=df_max, x='ts',
             element='step', stat="density", common_norm=False, ax=ax)
    pdf.savefig()
    


In [ ]:
# Apply FDR correction to each column
from statsmodels.stats import multitest
        
# Apply FDR correction to each numeric column
def apply_fdr_correction(series):
    _, p_adjusted, _, _ = multitest.multipletests(series, method='fdr_bh')
    return p_adjusted

        
coef_dfs = []
p_dfs = [] 
for subj in glms.participant.unique():
    subj_df = glms[glms.participant==subj]
    for chan in subj_df.chan.unique():
        
        test_df = subj_df[subj_df.chan==chan].groupby(['fband', 'params']).mean().reset_index()
        # Pivot the DataFrame
        coef_df = test_df.pivot(index='fband', columns='params', values='coef').reset_index()
        # Reset the index to make 'fband' a column again
        # If you don't want to reset the index, you can skip the next line
        coef_df = coef_df.reset_index(drop=True)
        # Rename the index column
        coef_df.columns.name = None

        # Pivot the DataFrame
        pval_df = test_df.pivot(index='fband', columns='params', values='p').reset_index()
        # Reset the index to make 'fband' a column again
        # If you don't want to reset the index, you can skip the next line
        pval_df = pval_df.reset_index(drop=True)
        # Rename the index column
        pval_df.columns.name = None

        # Select only numeric columns
        numeric_columns = pval_df.select_dtypes(include='number')
        
        fdr_df = numeric_columns.apply(apply_fdr_correction)

        fdr_df['fband'] = pval_df['fband']

        coef_df['participant'] = subj
        coef_df['chan'] = chan
        coef_df['region'] = subj_df[subj_df.chan==chan].region.unique()[0]
        coef_dfs.append(coef_df)

        fdr_df['participant'] = subj
        fdr_df['chan'] = chan
        fdr_df['region'] = subj_df[subj_df.chan==chan].region.unique()[0]
        p_dfs.append(fdr_df)

In [ ]:
all_p_df= pd.concat(p_dfs)
all_p_df = all_p_df[all_p_df.region!='Unknown']
numeric_columns = all_p_df.select_dtypes(include='number')

In [ ]:
all_coef_df = pd.concat(coef_dfs)
all_coef_df = all_coef_df[all_coef_df.region!='Unknown']


In [ ]:
g = sns.FacetGrid(all_coef_df, 
                  row='region', 
                  height=7.5, sharey=True, sharex=True)

g.map(sns.barplot, "fband", "zrpe",  errorbar='se', order=['theta', 'alpha', 'beta', 'slowgamma', 'hfa'])


In [ ]:
g = sns.FacetGrid(all_coef_df, 
                  row='region', 
                  height=7.5, sharey=True, sharex=True)

g.map(sns.barplot, "fband", "zpm",  errorbar='se', order=['theta', 'alpha', 'beta', 'slowgamma', 'hfa'])


## Load the dataframes for the time-resolved regression

In [ ]:
band_regress_dfs = []
for subj_id in progress_bar:
    filepath = f'/sc/arion/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
    for day in ['Day1']:
        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')
        for event in analysis_evs:
            for chan in elec_df.label.unique():
                if chan == 'Unknown':
                    continue
                else:
                    band_regress_df = pd.read_csv(f'{filepath}/{event}/dfs/{chan}_time_regressed_surr.csv')
                    band_regress_df = band_regress_df[band_regress_df.region!='Unknown']
                    # add subject - oops
                    band_regress_df['subj'] = subj_id
                    # check significance
                    band_regress_df['rpe_sig_upper'] = np.nan
                    band_regress_df['rpe_sig_lower'] = np.nan
                    band_regress_df['DPRIME_sig_upper'] = np.nan
                    band_regress_df['DPRIMe_sig_lower'] = np.nan
                    for band in band_regress_df.fband.unique():
                        band_regress_df['rpe_sig_upper'][band_regress_df.fband==band] = int(any(band_regress_df[band_regress_df.fband==band].rpe_p_upper_fdr<=0.025))
                        band_regress_df['rpe_sig_lower'][band_regress_df.fband==band] = int(any(band_regress_df[band_regress_df.fband==band].rpe_p_lower_fdr<=0.025))
                        band_regress_df['DPRIME_sig_upper'][band_regress_df.fband==band] = int(any(band_regress_df[band_regress_df.fband==band].DPRIME_p_upper_fdr<=0.025))
                        band_regress_df['DPRIMe_sig_lower'][band_regress_df.fband==band] = int(any(band_regress_df[band_regress_df.fband==band].DPRIME_p_lower_fdr<=0.025))
                    band_regress_dfs.append(band_regress_df)

In [ ]:
plot_df = band_regress_dfs[104]
for band in ['theta', 'alpha', 'beta', 'hfa']:
    for feature in ['rpe', 'DPRIME']:
        sig_time_pts_upper = plot_df[(plot_df.fband==band) & (plot_df[f'{feature}_p_upper_fdr']<=0.025)].ts
        sig_time_pts_lower = plot_df[(plot_df.fband==band) & (plot_df[f'{feature}_p_lower_fdr']<=0.025)].ts
        
        if (len(sig_time_pts_upper) > 0) | (len(sig_time_pts_lower) >0):
            fig, ax = plt.subplots(1, 1, figsize=(4, 4), dpi=300)
            sns.lineplot(data=plot_df[plot_df.fband==band], x='ts', y=f'{feature}')
            plt.plot(sig_time_pts_upper,
                     [plt.ylim()[1]]*len(sig_time_pts_upper), 'o')
            plt.plot(sig_time_pts_lower,
                     [plt.ylim()[0]]*len(sig_time_pts_lower), 'o')
            plt.suptitle(f'{band}_{feature}_{plot_df.region.unique()[0]}')

In [ ]:
features = ['rpe', 'DPRIME']
sig_chans = {f'{x}':[] for x in features}
sig_regions = {f'{x}':[] for x in features}
for plot_df in band_regress_dfs:
    for band in ['beta']:
        for feature in features:
            sig_time_pts_upper = plot_df[(plot_df.fband==band) & (plot_df[f'{feature}_p_upper_fdr']<=0.025)].ts
            sig_time_pts_lower = plot_df[(plot_df.fband==band) & (plot_df[f'{feature}_p_lower_fdr']<=0.025)].ts
            if (len(sig_time_pts_upper) > 0) | (len(sig_time_pts_lower) >0):
                sig_chans[feature].append(plot_df.chan.unique()[0]) 
                sig_regions[feature].append(plot_df.region.unique()[0])

In [ ]:
from collections import Counter
sig_chan_dfs = [] 
for feature in features:
    sig_chan_df = pd.DataFrame(columns=['region', 'count', 'feature'])
    sig_chan_df['region'] = Counter(sig_regions[feature]).keys()
    sig_chan_df['count'] = Counter(sig_regions[feature]).values()
    sig_chan_df['feature'] = feature 
    sig_chan_dfs.append(sig_chan_df) 
sig_chan_dfs = pd.concat(sig_chan_dfs)  

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(20, 4), dpi=300)
sns.barplot(data=sig_chan_dfs[sig_chan_dfs.region!='Unknown'], x='region', y='count', hue='feature')


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(20, 4), dpi=300)
sns.barplot(data=sig_chan_dfs[sig_chan_dfs.region!='Unknown'], x='region', y='count', hue='feature')
